# Martyna–Tuckerman vs exact nonperiodic limit

For a **compact** charge density in an increasingly large orthorhombic cell, MT should find (for sufficiently large cells) a consistent energy that should be very close to the nonperiodic limit. On the other hand, the periodic (regular) Hartree functional should show a slow convergence.
 

We consider **Hartree only** of a normalized Gaussian $\rho$.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from dftpy.constants import ENERGY_CONV, LEN_CONV
from dftpy.field import DirectField
from dftpy.formats.vasp import read_POSCAR
from dftpy.functional import Functional, TotalFunctional
from dftpy.functional.hartree import Hartree
from dftpy.functional.martyna_tuckerman import MartynaTuckerman
from dftpy.functional.pseudo import LocalPseudo
from dftpy.grid import DirectGrid
from dftpy.ions import Ions
from dftpy.math_utils import ecut2nr

HERE = Path.cwd().resolve()
for DATA in (HERE.parent / "DATA", HERE / "DATA"):
    if (DATA / "mg.lda.recpot").is_file():
        break
else:
    raise FileNotFoundError("Expected examples/DATA with mg.lda.recpot; cd to examples/notebooks")

Ha_to_eV = ENERGY_CONV["Hartree"]["eV"]
A_to_Bohr = LEN_CONV["Angstrom"]["Bohr"]

## 1) Hartree: $E_{\mathrm{MT}}$ for a Gaussian $\rho(r)$

Same normalized Gaussian width $\sigma$ in **physical** length units; grid resolution from **`ecut`** (spacing $\propto 1/\sqrt{E_{\mathrm{cut}}}$). Box side $L$ grows.

In [ ]:
ECUT = 50.0  # lower if too slow; raise toward 90 to match AIMD notebook
SIGMA_BOHR = 1.25

L_ang = np.array([14.0, 20.0, 28.0, 40.0, 56.0])
delta2_ev = []
delta_ev = []

for L in L_ang:
    Lb = L * A_to_Bohr
    lattice = np.eye(3, dtype=np.float64) * Lb
    nr = ecut2nr(ecut=ECUT, lattice=lattice)
    origin = np.array([Lb / 2.0, Lb / 2.0, Lb / 2.0], dtype=np.float64)
    grid = DirectGrid(lattice=lattice, nr=nr, full=True, origin=origin)
   
    mt = MartynaTuckerman(grid,alpha=np.sqrt(7/Lb)) # here can use a wide range of alphas
    
    rr = grid.r_mic**2
    rho = DirectField(grid=grid, griddata_3d=(2.0*np.pi*SIGMA_BOHR**2)**(-3.0/2.0)*np.exp(-rr / (2.0 * SIGMA_BOHR**2)))
    rho /= rho.integral()

    e_exact = 1.0/2.0/SIGMA_BOHR/np.sqrt(np.pi)
    e_mt = Hartree(mt=mt)(rho, calcType={"E"}).energy
    e_pw = Hartree()(rho, calcType={"E"}).energy
    d = e_mt - e_exact
    d2 = e_pw - e_exact
    delta_ev.append(d * Ha_to_eV)
    delta2_ev.append(d2 * Ha_to_eV)

delta_ev = np.asarray(delta_ev)
delta2_ev = np.asarray(delta2_ev)

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.semilogy(L_ang, np.abs(delta_ev), "o-", label=r"$|E_{\mathrm{MT}}^{\mathrm{(H)}} - E_{\mathrm{exact}}|$ (eV)")
ax.semilogy(L_ang, np.abs(delta2_ev), "o-", label=r"$|E_{\mathrm{PW}}^{\mathrm{(H)}} - E_{\mathrm{exact}}|$ (eV)")
ax.set_xlabel(r"Cubic box side $L$ (Å)")
ax.set_ylabel("|Δ Hartree| (eV)")
ax.grid(True, which="both", ls=":")
ax.legend()
plt.tight_layout()
plt.show()

print("L (Å)       ΔE_mt (eV)       ΔE_pw (eV)")
for i, L in enumerate(L_ang):
    print(f"{L:6.1f}    {delta_ev[i]:+.6e}, {delta2_ev[i]:+.6e}")

Given MT Alpha:  0.5143817699743262
wg 1665.7805984665097
Given MT Alpha:  0.4303626653153645
wg 3399.660351922564
Given MT Alpha:  0.36372283766758484
wg 6663.4340091230415
Given MT Alpha:  0.3043123590140108
wg 13598.953032605783
Given MT Alpha:  0.2571908849871631
wg 26654.04766595298
